In [1]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import json
from peft import prepare_model_for_kbit_training

g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [ ]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading checkpoint shards: 100%|██████████| 5/5 [01:44<00:00, 20.92s/it]


In [ ]:
dataset = load_dataset(
    "json",
    data_files="./V5D/V5D.jsonl",
)

train_dataset = dataset["train"]

print("Training samples:", len(train_dataset))

Generating train split: 65385 examples [00:00, 102007.67 examples/s]


Training samples: 65385


In [10]:
# ==========================================================
# V5B — Communication Strategy Formatting
# ==========================================================

import json

# ----------------------------------------------------------
# Prepare Model
# ----------------------------------------------------------

model = prepare_model_for_kbit_training(model)

model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.config.use_cache = False


# ==========================================================
# V5B SYSTEM PROMPT
# ==========================================================

SYSTEM_PROMPT = """You are an expert assistant communication strategy prediction model.

Your task is to predict HOW the assistant should communicate based only on the provided relationship context and conversation.

You are NOT generating the assistant's final response.

Rules:
- Predict only the communication strategy.
- Do NOT generate the final assistant reply.
- Do NOT summarize the conversation.
- Do NOT extract memories.
- Do NOT predict the assistant's objective.
- Do NOT invent facts.
- Do NOT infer unsupported communication preferences.
- Select one appropriate tone.
- Select one appropriate communication style.
- Select one appropriate detail level.
- Provide one concise communication approach.

Return ONLY valid JSON in exactly this format:

{
  "tone": "",
  "communication_style": "",
  "detail_level": "",
  "approach": ""
}

Field Definitions:

- tone: The emotional or professional tone the assistant should use.
- communication_style: The overall communication style.
- detail_level: Must be "Brief", "Moderate", or "Detailed".
- approach: A concise description of HOW the assistant should communicate.

Do NOT generate the final response.
Do NOT summarize the conversation.
Do NOT extract memories.
Do NOT predict objectives.
Do NOT provide multiple strategies.

Return only the JSON object.
Do not include markdown or extra text.
"""


# ==========================================================
# FORMATTING FUNCTION
# ==========================================================

def formatting_func(example):

    input_json = json.dumps(
        example["input"],
        ensure_ascii=False,
        separators=(",", ":")
    )

    user_input = (
        example["instruction"]
        + "\n\n"
        + input_json
    )

    assistant_output = json.dumps(
        {
            "tone": example["output"]["tone"],
            "communication_style": example["output"]["communication_style"],
            "detail_level": example["output"]["detail_level"],
            "approach": example["output"]["approach"]
        },
        ensure_ascii=False,
        separators=(",", ":")
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_input
        },
        {
            "role": "assistant",
            "content": assistant_output
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )


# ==========================================================
# TEST ONLY — USE EXISTING DATASET
# ==========================================================

print("\n" + "=" * 70)
print("V5B FORMATTING TEST")
print("=" * 70)

test_text = formatting_func(train_dataset[0])

print(test_text)

print("\n" + "=" * 70)
print("V5B FORMATTING TEST COMPLETE")
print("=" * 70)


V5B FORMATTING TEST
<|im_start|>system
You are an expert assistant communication strategy prediction model.

Your task is to predict HOW the assistant should communicate based only on the provided relationship context and conversation.

You are NOT generating the assistant's final response.

Rules:
- Predict only the communication strategy.
- Do NOT generate the final assistant reply.
- Do NOT summarize the conversation.
- Do NOT extract memories.
- Do NOT predict the assistant's objective.
- Do NOT invent facts.
- Do NOT infer unsupported communication preferences.
- Select one appropriate tone.
- Select one appropriate communication style.
- Select one appropriate detail level.
- Provide one concise communication approach.

Return ONLY valid JSON in exactly this format:

{
  "tone": "",
  "communication_style": "",
  "detail_level": "",
  "approach": ""
}

Field Definitions:

- tone: The emotional or professional tone the assistant should use.
- communication_style: The overall comm

In [12]:
# Printing  TOKEN ,DISTRIBUTION ,FORMATTING ,LONGEST ,SAMPLES of this dataset (important thing before training)

import time
from statistics import mean, median

def analyze_dataset(name, dataset, tokenizer, formatting_func):
    print("\n" + "="*70)
    print(name)
    print("="*70)

    train = dataset["train"]

    lengths = []
    format_times = []

    start_total = time.time()

    for i, sample in enumerate(train):
        t1 = time.time()

        text = formatting_func(sample)

        t2 = time.time()
        format_times.append(t2 - t1)

        tokens = tokenizer(text, add_special_tokens=True)["input_ids"]
        lengths.append(len(tokens))

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{len(train)}")

    total_time = time.time() - start_total

    print("\n----- BASIC -----")
    print("Samples              :", len(train))
    print("Average Tokens       :", round(mean(lengths),2))
    print("Median Tokens        :", median(lengths))
    print("Minimum Tokens       :", min(lengths))
    print("Maximum Tokens       :", max(lengths))

    print("\n----- TOKEN DISTRIBUTION -----")
    print(">256 tokens          :", sum(x > 256 for x in lengths))
    print(">512 tokens          :", sum(x > 512 for x in lengths))
    print(">1024 tokens         :", sum(x > 1024 for x in lengths))
    print(">2048 tokens         :", sum(x > 2048 for x in lengths))
    print(">4096 tokens         :", sum(x > 4096 for x in lengths))

    print("\n----- FORMATTING -----")
    print("Formatting Time      :", round(total_time,2), "sec")
    print("Average/sample       :", round(mean(format_times)*1000,3), "ms")
    print("Samples/sec          :", round(len(train)/total_time,2))

    print("\n----- LONGEST SAMPLES -----")
    top = sorted(enumerate(lengths), key=lambda x: x[1], reverse=True)[:10]

    for idx, tok in top:
        print(f"Sample {idx:6d} : {tok} tokens")

    return lengths


In [13]:
old_dataset = load_dataset(
    "json",
    data_files= r"./V5B-65K-Isha.jsonl",
  
)

# new_dataset = load_dataset(
#     "json",
#     data_files=r"./newV4a/finalV4A.jsonl",
  
# )


old_lengths = analyze_dataset(
    "OLD DATASET",
    old_dataset,
    tokenizer,
    formatting_func
)

# new_lengths = analyze_dataset(
#     "NEW V4A DATASET",
#     new_dataset,
#     tokenizer,
#     formatting_func
# )


OLD DATASET
Processed 5000/65385
Processed 10000/65385
Processed 15000/65385
Processed 20000/65385
Processed 25000/65385
Processed 30000/65385
Processed 35000/65385
Processed 40000/65385
Processed 45000/65385
Processed 50000/65385
Processed 55000/65385
Processed 60000/65385
Processed 65000/65385

----- BASIC -----
Samples              : 65385
Average Tokens       : 530.72
Median Tokens        : 522
Minimum Tokens       : 363
Maximum Tokens       : 1082

----- TOKEN DISTRIBUTION -----
>256 tokens          : 65385
>512 tokens          : 36503
>1024 tokens         : 1
>2048 tokens         : 0
>4096 tokens         : 0

----- FORMATTING -----
Formatting Time      : 123.52 sec
Average/sample       : 0.246 ms
Samples/sec          : 529.33

----- LONGEST SAMPLES -----
Sample  12884 : 1082 tokens
Sample   4830 : 987 tokens
Sample  15461 : 986 tokens
Sample  51429 : 982 tokens
Sample  13881 : 979 tokens
Sample  18439 : 951 tokens
Sample  19088 : 950 tokens
Sample  18849 : 947 tokens
Sample  142

In [ ]:
import json
from collections import defaultdict, Counter

FILE = r"V5B-65K-Isha.jsonl"

def value_type(value):
    if value is None:
        return "null"
    if isinstance(value, dict):
        return "object"
    if isinstance(value, list):
        return "array"
    if isinstance(value, str):
        return "string"
    if isinstance(value, bool):
        return "boolean"
    if isinstance(value, (int, float)):
        return "number"
    return type(value).__name__


with open(FILE, "r", encoding="utf-8") as f:
    first = f.read(1)

if first == "[":
    with open(FILE, "r", encoding="utf-8") as f:
        records = json.load(f)
else:
    records = []
    with open(FILE, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except Exception as e:
                    print("Invalid JSON:", line_no, e)


types = defaultdict(Counter)

for record in records:

    types["instruction"][value_type(record.get("instruction"))] += 1

    inp = record.get("input", {})

    if isinstance(inp, dict):
        for field in ["domain", "relationship", "conversation"]:
            types[f"input.{field}"][value_type(inp.get(field))] += 1

    out = record.get("output", {})

    if isinstance(out, dict):
        for field in [
            "tone",
            "communication_style",
            "detail_level",
            "approach"
        ]:
            types[f"output.{field}"][value_type(out.get(field))] += 1


print("\nTYPE DISTRIBUTION")
print("=" * 60)

for field, counter in types.items():
    print(f"\n{field}")
    for typ, count in counter.items():
        print(f"   {typ}: {count}")


TYPE DISTRIBUTION

instruction
   string: 65454

input.domain
   string: 65454

input.relationship
   string: 65454

input.conversation
   string: 65454

output.tone
   string: 65385
   null: 69

output.communication_style
   string: 65385
   null: 69

output.detail_level
   string: 65385
   null: 69

output.approach
   string: 65385
   null: 69


In [ ]:
import json

INPUT_FILE = r"V5B-65K-Isha.jsonl"
OUTPUT_FILE = r"V5B-65K-Isha-clean.jsonl"

required_fields = [
    "tone",
    "communication_style",
    "detail_level",
    "approach"
]

total = 0
clean = 0
removed = 0

with open(INPUT_FILE, "r", encoding="utf-8") as infile, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:

    for line_number, line in enumerate(infile, 1):

        line = line.strip()

        if not line:
            continue

        total += 1

        try:
            record = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"❌ Invalid JSON at line {line_number}: {e}")
            removed += 1
            continue

        output = record.get("output")

        # Check output exists and all 4 fields are valid strings
        valid = (
            isinstance(output, dict)
            and all(
                isinstance(output.get(field), str)
                and output.get(field).strip()
                for field in required_fields
            )
        )

        if not valid:
            removed += 1
            print(f"❌ Removing invalid record at line {line_number}")
            continue

        # Write clean JSONL record
        outfile.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

        clean += 1


print("\n" + "=" * 60)
print("V5B CLEANING COMPLETE")
print("=" * 60)

print("Original records :", total)
print("Clean records    :", clean)
print("Removed records  :", removed)
print("Output file      :", OUTPUT_FILE)

❌ Removing invalid record at line 1373
❌ Removing invalid record at line 1374
❌ Removing invalid record at line 1375
❌ Removing invalid record at line 1376
❌ Removing invalid record at line 1377
❌ Removing invalid record at line 4470
❌ Removing invalid record at line 4471
❌ Removing invalid record at line 4472
❌ Removing invalid record at line 4473
❌ Removing invalid record at line 4474
❌ Removing invalid record at line 11109
❌ Removing invalid record at line 11110
❌ Removing invalid record at line 11111
❌ Removing invalid record at line 11112
❌ Removing invalid record at line 11113
❌ Removing invalid record at line 11114
❌ Removing invalid record at line 11115
❌ Removing invalid record at line 11116
❌ Removing invalid record at line 11117
❌ Removing invalid record at line 11118
❌ Removing invalid record at line 11119
❌ Removing invalid record at line 11120
❌ Removing invalid record at line 11121
❌ Removing invalid record at line 11122
❌ Removing invalid record at line 11123
❌ Removing